In [41]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

sample = pd.read_csv("../../data/processed/sample.csv", sep="|")
sample.head()

,lineID,day,pid,adFlag,availability,competitorPrice,click,basket,order,price,...,genericProduct,salesIndex,category,rrp,campaignIndex_A,campaignIndex_B,campaignIndex_C,price_diff_competitor,price_ratio_competitor,price_pct_diff_competitor
0,978899,39,9624,0,1,17.19,1,0,0,19.89,...,0,40,17.0,21.51,0,0,0,2.70,1.16,15.71
1,1267035,47,3969,1,1,18.13,1,0,0,20.85,...,0,40,9.0,26.07,0,0,0,2.72,1.15,15.00
2,297914,14,16633,0,1,15.06,0,0,1,16.45,...,0,40,132.0,23.98,0,1,0,1.39,1.09,9.23
3,2554963,87,20147,0,1,4.36,1,0,0,5.17,...,0,53,28.0,5.45,0,0,0,0.81,1.19,18.58
4,2739211,92,14326,0,1,6.12,0,0,1,6.22,...,0,53,3.0,6.55,0,0,0,0.10,1.02,1.63


In [42]:
#target variable
Y = sample["order"]

#drop unnessacary tables:
X = sample.drop(columns=["order", "lineID", "revenue", "click", "basket", "group", "unit", "pharmForm", 'price_diff_competitor', 'price_pct_diff_competitor'])
X.head()

,day,pid,adFlag,availability,competitorPrice,price,manufacturer,content,genericProduct,salesIndex,category,rrp,campaignIndex_A,campaignIndex_B,campaignIndex_C,price_ratio_competitor
0,39,9624,0,1,17.19,19.89,17,30,0,40,17.0,21.51,0,0,0,1.16
1,47,3969,1,1,18.13,20.85,556,80,0,40,9.0,26.07,0,0,0,1.15
2,14,16633,0,1,15.06,16.45,168,50,0,40,132.0,23.98,0,1,0,1.09
3,87,20147,0,1,4.36,5.17,101,30,0,53,28.0,5.45,0,0,0,1.19
4,92,14326,0,1,6.12,6.22,119,120,0,53,3.0,6.55,0,0,0,1.02


Feature Selection

Univariate Feature Selection

In [43]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['day', 'pid', 'adFlag', 'availability', 'competitorPrice', 'price', 'manufacturer', 'content', 'genericProduct', 'salesIndex', 'category', 'rrp', 'campaignIndex_A', 'campaignIndex_B', 'campaignIndex_C', 'price_ratio_competitor']
Categorical: []


In [44]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
    ]
)

# -------------------------
# 5) Build pipeline:
# preprocess -> feature selection -> model
# -------------------------
pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("select", SelectKBest(score_func=f_classif, k=20)),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# -------------------------
# 6) Train/test split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# 7) Fit model
# -------------------------
pipeline.fit(X_train, y_train)

# -------------------------
# 8) Evaluate
# -------------------------
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

C:\Users\saidf\PycharmProjects\analyticalProjectProductPricing\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:782: UserWarning: k=20 is greater than n_features=16. All the features will be returned.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.80      0.62      0.70     29766
           1       0.34      0.56      0.42     10234

    accuracy                           0.60     40000
   macro avg       0.57      0.59      0.56     40000
weighted avg       0.68      0.60      0.63     40000

ROC-AUC: 0.6222612053123213


C:\Users\saidf\PycharmProjects\analyticalProjectProductPricing\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [45]:
scores = pipeline.named_steps["select"].scores_

feature_scores = pd.DataFrame({
    "feature": feature_names,
    "score": scores
}).sort_values("score", ascending=False)

print(feature_scores.head(30))

ValueError: All arrays must be of the same length

In [38]:
from sklearn.model_selection import cross_val_score

k_values = [5, 10, 20, 30, 50]

results = []

for k in k_values:
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ])

    scores = cross_val_score(
        pipe, X_train, y_train,
        cv=3,
        scoring="roc_auc",
        n_jobs=-1
    )

    results.append({
        "k": k,
        "mean_auc": scores.mean(),
        "std_auc": scores.std()
    })

results_df = pd.DataFrame(results).sort_values("mean_auc", ascending=False)
print(results_df)

    k  mean_auc   std_auc
1  10  0.616935  0.002261
3  30  0.610564  0.005657
2  20  0.610564  0.005657
4  50  0.610564  0.005657
0   5  0.605220  0.000355
